# MolScribe single-image inference demo

This evaluator-facing notebook performs genuine inference on a newly uploaded PNG or JPEG using the project-trained augmented MolScribe checkpoint. It does not retrain the model or rerun the 5,088-image benchmark.

Requirements:

- Google Colab with a **T4 GPU** runtime.
- The augmented checkpoint in the Google Drive path used below.
- `molscribe_gradio_app.py` from the project's `demo` folder.

Run all cells in order. The final cell prints a temporary Gradio link through which the evaluator can upload one structure image.

## 1. Create the isolated inference environment

In [ ]:
import hashlib, subprocess, sys
from pathlib import Path

assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'Select Runtime > Change runtime type > T4 GPU'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.10'], check=True)
subprocess.run(['uv', 'venv', '/content/molscribe_demo_env', '--python', '3.10', '--clear'], check=True)
PY = '/content/molscribe_demo_env/bin/python'
print('Isolated Python:', subprocess.check_output([PY, '--version'], text=True).strip())

## 2. Install pinned dependencies

In [ ]:
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--extra-index-url', 'https://download.pytorch.org/whl/cu117', 'torch==1.13.1+cu117', 'torchvision==0.14.1+cu117'], check=True)
deps = ['numpy==1.23.5', 'pandas==1.5.3', 'scipy==1.10.1', 'matplotlib==3.7.5', 'opencv-python-headless==4.5.5.64', 'Pillow==9.5.0', 'SmilesPE==0.0.3', 'rdkit==2022.9.5', 'albumentations==1.1.0', 'timm==0.4.12', 'transformers==4.30.2', 'huggingface-hub==0.16.4', 'tensorboardX==2.6.2.2', 'torchtext==0.5.0', 'configargparse==1.7', 'pyonmttok==1.37.1', 'PyYAML==6.0.1']
subprocess.run(['uv', 'pip', 'install', '--python', PY, *deps], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--no-deps', 'OpenNMT-py==2.2.0'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', PY, 'gradio==4.44.1'], check=True)
subprocess.run([PY, '-c', 'import torch, gradio; print(torch.__version__, torch.cuda.is_available(), gradio.__version__)'], check=True)

## 3. Install the exact MolScribe revision

In [ ]:
REPO = Path('/content/MolScribe')
COMMIT = '7296a30413eb55436702011efdff78131f66d162'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/thomas0809/MolScribe.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--no-deps', '-e', str(REPO)], check=True)
print('MolScribe commit verified:', COMMIT)

## 4. Load and verify the project-trained checkpoint

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
CKPT = Path('/content/drive/MyDrive/AIDD_Mini_Project_3/molscribe_augmented_probe/swin_base_transformer_best.pth')
EXPECTED_SHA256 = '20bba8c832365f1b708cfe7738b5dfd5f90b025bf6a4619288240e0c6f313741'
assert CKPT.is_file(), f'Checkpoint not found: {CKPT}'
digest = hashlib.sha256()
with CKPT.open('rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_SHA256, 'Checkpoint SHA-256 mismatch'
print('Augmented checkpoint verified:', CKPT.stat().st_size, 'bytes')

## 5. Upload the demonstration application

Upload only `molscribe_gradio_app.py` from the submission folder's `demo` directory.

In [ ]:
from google.colab import files

uploaded = files.upload()
assert list(uploaded) == ['molscribe_gradio_app.py'], 'Upload only molscribe_gradio_app.py'
APP = Path('/content/molscribe_gradio_app.py')
assert APP.is_file()
print('Demo application ready:', APP)

## 6. Start genuine single-image inference

The first model load can take several minutes because the checkpoint is approximately 1.1 GB. Keep this cell running while the evaluator uses the printed Gradio link, and stop it after the demonstration. Use only non-sensitive structure images because `--share` creates a temporary public URL.

In [ ]:
subprocess.run([
    PY, str(APP),
    '--checkpoint', str(CKPT),
    '--molscribe-root', str(REPO),
    '--share',
], check=True)